In [85]:
import pandas as pd
df =pd.read_excel("./1.xlsx")

In [86]:
df['商品售后'] = df['商品售后'].replace('', '无').fillna('无')

In [87]:
keywords=['10001911955390','10001392472592','10001456659976','10001375091184','10001172504206','10001299864714','10001152241224','10000932727110','10000964252250','10000806375949','10000746057283','10000596166903','10000630009137','10000560103600','10000560064538','10000559631447','10000546655452','10000527984279','10000560103600','10000400290142','10000382097166']
# 10000229167469,10000229407532,10000265046381,10000270409428,10000269553640,10000273809516
df['商品编码(平台)'].isin(keywords)
df['商品编码(平台)'] = df['商品编码(平台)'].astype(str)


#删除其他行
df.drop(index= df[ df['商品编码(平台)'].isin(keywords)==False].index,inplace=True)
df.reset_index(drop=True,inplace= True)

influencer_id = ['洋奢女子塑身臻选','YANGSHE塑形服饰','洋奢塑形美体优选','洋奢精选内衣好物', '洋奢时尚内衣优选','洋奢女装内衣服饰','洋奢塑形服饰内衣','洋奢上善臻选美体内衣服饰','洋奢内衣好物精选', '-']
other_influencers = df[~df['带货账号昵称'].isin(influencer_id)]
print(f"\n其他达人信息:{other_influencers[['带货账号昵称']].drop_duplicates()}")
df = df[df['带货账号昵称'].isin(influencer_id)]


其他达人信息:Empty DataFrame
Columns: [带货账号昵称]
Index: []


In [88]:
df["是否刷单"] = df["商家备注"].str.contains("物品", na=False)

In [89]:
df = df[~df["是否刷单"]].copy()

In [90]:
df.rename(columns={"订单下单时间":"下单时间"},inplace=True)

In [91]:
df["支付时间"] = pd.to_datetime(df["支付时间"],format = "%Y-%m-%d %H:%M:%S",errors='coerce')
df["支付时间"] = df["支付时间"].fillna(pd.to_datetime('1970-01-01'))

In [92]:
df["订单发货时间"] = pd.to_datetime(df["订单发货时间"],format = "%Y-%m-%d %H:%M:%S",errors='coerce')
df["订单发货时间"] = df["订单发货时间"].fillna(pd.to_datetime('1970-01-01'))

In [93]:
df['发货时长（小时）'] = pd.to_datetime(df['订单发货时间']) - pd.to_datetime(df['支付时间'])

In [94]:
df['发货时长（小时）'] = df['发货时长（小时）'].dt.total_seconds() / 3600

In [95]:
df['发货时长（小时）'] = df['发货时长（小时）'].clip(lower=0)

In [96]:
df['发货时长（小时）']

0       0.029722
1       0.055556
2       0.125278
3       0.000000
4       0.049722
5       0.194722
6       0.013889
7       0.016944
8       0.146111
9       0.227222
10      1.626389
11      0.039722
12      0.047778
13      0.091667
14      0.124167
15      0.186667
16      0.201389
17      0.201389
18      0.000000
19      0.230000
20      0.000000
21      0.017222
22      0.011667
23      0.030556
24      0.072778
25      0.084722
26      0.114167
27      0.111389
28      0.127778
29      0.160278
          ...   
4669    0.136944
4670    0.019722
4671    0.030556
4672    0.000000
4673    0.248889
4674    0.131111
4675    0.236667
4676    0.020556
4677    0.000000
4678    0.000000
4679    0.020000
4680    0.141111
4681    0.150278
4682    0.000000
4683    0.039722
4684    0.119444
4685    0.120833
4686    0.174444
4687    0.012500
4688    0.205556
4689    0.207778
4690    0.000000
4691    0.080000
4692    0.061389
4693    0.239167
4694    0.245278
4695    0.056389
4696    0.2483

In [97]:
if '\t' in df.loc[0,'下单时间']:
    df['下单时间'] = df['下单时间'].str[1:]

In [98]:
from datetime import datetime
df["下单时间"] = pd.to_datetime(df["下单时间"],format = "%Y-%m-%d %H:%M:%S")
df["下单时间"] = df["下单时间"].map(lambda x: x.strftime('%Y-%m-%d'))
df["销售金额"]= (df['商品数量'] * df['商品价格(单件)']) + (df['订单运费'])
df1 = pd.read_excel("商品编码.xlsx")
df1.rename(columns={"商品名":"产品名称","商家编码":"SKU编码(自定义)"},inplace=True)
df1 = df1[["SKU编码(自定义)","产品名称"]]
df= pd.merge(df,df1,on="SKU编码(自定义)",how="left")

In [99]:
# 缺码表
lackCode = df[df['产品名称'].isnull()]['SKU编码(自定义)'].value_counts()
lackCode

Series([], Name: SKU编码(自定义), dtype: int64)

In [100]:
lackCode.to_excel('缺码表.xlsx')

In [101]:
df['订单数'] = df['商品数量']

In [102]:
from datetime import datetime
#df["支付时间"] = df["支付时间"].fillna(pd.to_datetime('1970-01-01'))
df["支付时间"] = pd.to_datetime(df["支付时间"],format = "%Y-%m-%d %H:%M:%S",errors='coerce')
df["支付时间"] = df["支付时间"].fillna(pd.to_datetime('1970-01-01'))
df["支付时间"] = df["支付时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [103]:
df["支付时间"]

0       2026-11-18
1       2026-11-18
2       2026-11-18
3       1970-01-01
4       2026-11-18
5       2026-11-18
6       2026-11-18
7       2026-11-18
8       2026-11-18
9       2026-11-18
10      2026-11-18
11      2026-11-18
12      2026-11-18
13      2026-11-18
14      2026-11-18
15      2026-11-18
16      2026-11-18
17      2026-11-18
18      1970-01-01
19      2026-11-18
20      2026-11-18
21      2026-11-18
22      2026-11-18
23      2026-11-18
24      2026-11-18
25      2026-11-18
26      2026-11-18
27      2026-11-18
28      2026-11-18
29      2026-11-18
           ...    
4316    2026-10-28
4317    2026-10-28
4318    2026-10-28
4319    1970-01-01
4320    2026-10-28
4321    2026-10-28
4322    2026-10-28
4323    2026-10-28
4324    1970-01-01
4325    1970-01-01
4326    2026-10-28
4327    2026-10-28
4328    2026-10-28
4329    1970-01-01
4330    2026-10-28
4331    2026-10-28
4332    2026-10-28
4333    2026-10-28
4334    2026-10-28
4335    2026-10-28
4336    2026-10-28
4337    1970

In [104]:
df["下单单量"] = df['订单数']
df["支付单量"] = 0
df["发货单量"] = 0
df["发货前退款单量"] = 0

In [105]:
for i in range(0,len(df)):
    if df.loc[i,"支付时间"] != "1970-01-01":
        df.loc[i,"支付单量"] = df.loc[i,'订单数']
        df.loc[i,"支付金额"] = df.loc[i,'订单实际收款金额']

In [106]:
for i in range(0,len(df)):
    if df.loc[i,"商品发货"] == "已发货" :
        df.loc[i,"发货单量"] = df.loc[i,'订单数']

In [107]:
for i in range(0,len(df)):
    if df.loc[i,"支付时间"] != "1970-01-01" and  df.loc[i,"商品发货"] == "未发货" and df.loc[i,"商品售后"] == "退款完成":
        df.loc[i,"发货前退款单量"] = df.loc[i,'订单数']

In [108]:
df["已签收"] = 0
df["发货中1"] = 0
df["售后中"] = 0
df["已退款"] = 0
df["预计签收单量"] = 0
df["发货中2"] = 0
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "已完成" and ((df.loc[i,'商品售后'] == "无") or (df.loc[i,'商品售后'] == "用户取消申请") or (df.loc[i,'商品售后'] == "平台处理完成，售后关闭") or (df.loc[i,'商品售后'] == "NaN")):
         df.loc[i,"已签收"] = df.loc[i,'订单数']
    elif df.loc[i,"订单状态"] == "已发货" and ((df.loc[i,'商品售后'] == "无") or (df.loc[i,'商品售后'] == "用户取消申请") or (df.loc[i,'商品售后'] == "平台处理完成，售后关闭") ):
         df.loc[i,"发货中1"] = df.loc[i,'订单数']
    elif df.loc[i,'商品售后'] =="待买家退货" or df.loc[i,'商品售后'] =="待商家收货" or df.loc[i,'商品售后'] =="售后协商中" or df.loc[i,'商品售后'] =="待商家受理" :
        df.loc[i,"售后中"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"商品发货"] == "已发货" and (df.loc[i,'商品售后'] == "退货退款完成" or df.loc[i,'商品售后'] == "退款完成"):
        df.loc[i,"已退款"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"商品发货"] == "已发货" and (df.loc[i,'商品售后'] == "无" or df.loc[i,'商品售后'] == "用户取消申请" or df.loc[i,'商品售后'] == "平台处理完成，售后关闭"):
         df.loc[i,"预计签收单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "已取消" and  df.loc[i,"商品发货"] == "已发货" and (df.loc[i,'商品售后'] == "用户取消申请" or df.loc[i,'商品售后'] == "平台处理完成，售后关闭" ) :
        df.loc[i,"发货中2"] = df.loc[i,'订单数']

In [109]:
df["订单发货时间"] = pd.to_datetime(df["订单发货时间"],format = "%Y-%m-%d %H:%M:%S",errors='coerce')
df["订单发货时间"] = df["订单发货时间"].fillna(pd.to_datetime('1970-01-01'))
df["订单发货时间"]

0      2026-11-18 09:57:30
1      2026-11-18 09:57:30
2      2026-11-18 09:57:30
3      1970-01-01 00:00:00
4      2026-11-18 09:27:09
5      2026-11-18 09:11:19
6      2026-11-18 08:56:47
7      2026-11-18 08:56:47
8      2026-11-18 08:56:47
9      2026-11-18 08:41:39
10     2026-11-18 09:59:18
11     2026-11-18 08:12:13
12     2026-11-18 08:12:13
13     2026-11-18 08:12:13
14     2026-11-18 08:12:13
15     2026-11-18 08:12:13
16     2026-11-18 08:12:13
17     2026-11-18 08:12:13
18     1970-01-01 00:00:00
19     2026-11-18 08:12:13
20     1970-01-01 00:00:00
21     2026-11-18 07:56:59
22     2026-11-18 07:56:59
23     2026-11-18 07:56:59
24     2026-11-18 07:56:59
25     2026-11-18 07:56:59
26     2026-11-18 07:56:59
27     2026-11-18 07:56:59
28     2026-11-18 07:56:59
29     2026-11-18 07:56:59
               ...        
4316   2026-10-28 08:49:39
4317   2026-10-28 08:35:12
4318   2026-10-28 08:35:12
4319   1970-01-01 00:00:00
4320   2026-10-28 08:35:12
4321   2026-10-28 08:19:21
4

In [110]:
df["单件支付单量"] = 0
df["双件支付单量"] = 0
df["单件签收单量"] = 0
df["双件签收单量"] = 0
df["单件发货前退款单量"] = 0
df["双件发货前退款单量"] = 0

In [111]:
for i in range(0,len(df)):
    if df.loc[i,"商品价格(单件)"] == 49.9 :
        df.loc[i,"单件支付单量"] = df.loc[i,'支付单量']

In [112]:
for i in range(0,len(df)):
    if df.loc[i,"商品价格(单件)"] == 59.9 :
        df.loc[i,"双件支付单量"] = df.loc[i,'支付单量']

In [113]:
for i in range(0,len(df)):
    if df.loc[i,"商品价格(单件)"] == 49.9 :
        df.loc[i,"单件签收单量"] = df.loc[i,'预计签收单量']

In [114]:
for i in range(0,len(df)):
    if df.loc[i,"商品价格(单件)"] == 59.9 :
        df.loc[i,"双件签收单量"] = df.loc[i,'预计签收单量']

In [115]:
for i in range(0,len(df)):
    if df.loc[i,"商品价格(单件)"] == 49.9 :
        df.loc[i,"单件发货前退款单量"] = df.loc[i,'发货前退款单量']

In [116]:
for i in range(0,len(df)):
    if df.loc[i,"商品价格(单件)"] == 59.9 :
        df.loc[i,"双件发货前退款单量"] = df.loc[i,'发货前退款单量']

In [117]:
import numpy as np
df['销售金额'].fillna(0, inplace=True)

df['运费险预计投保费用'] = pd.to_numeric(df['运费险预计投保费用'], errors='coerce')

result = pd.pivot_table(df,index=['下单时间',"产品名称"] , values = ['下单单量','支付单量','发货单量','已签收','发货中1','发货中2','售后中','已退款','发货前退款单量',"预计签收单量",'运费险预计投保费用','支付金额','发货时长（小时）','单件支付单量','双件支付单量','单件签收单量','双件签收单量','单件发货前退款单量','双件发货前退款单量'] , aggfunc = np.sum)
result = result.reset_index()



result['运费险预计投保费用'] = np.where(result['运费险预计投保费用'].isna() | (result['运费险预计投保费用'] == 0), '关闭', '开')


result = result[['下单时间',"产品名称",'下单单量','支付单量','发货单量','已签收','发货中1','发货中2','售后中','已退款','发货前退款单量',"预计签收单量",'运费险预计投保费用','支付金额','发货时长（小时）','单件支付单量','双件支付单量','单件签收单量','双件签收单量','单件发货前退款单量','双件发货前退款单量']  ]

In [118]:
#拼接店铺消耗表
df_consume = pd.read_excel("店铺消耗微信豆25号.xlsx")
df_consume = df_consume[['日期','产品名称','累计消耗金额','成交订单数','成交金额']]
temp = df_consume.日期.str.extract('(\d+).*?(\d+).*?(\d+)') 
df_consume['下单时间'] = temp[0] + "-" + temp[1] + '-' + temp[2]
del df_consume['日期']
df_consume = pd.pivot_table(df_consume,index=['下单时间','产品名称'] , values = ['累计消耗金额','成交订单数','成交金额'] , aggfunc = np.sum)
df_consume = df_consume.reset_index()
df_consume["下单时间"] = pd.to_datetime(df_consume["下单时间"],format = "%Y-%m-%d")
df_consume["下单时间"] = df_consume["下单时间"].map(lambda x: x.strftime('%Y-%m-%d'))
df_consume

,下单时间,产品名称,成交订单数,成交金额,累计消耗金额
0,2026-01-03,2588#刺绣收腹提臀裤,0.0,0.00,27.01
1,2026-01-04,2588#刺绣收腹提臀裤,0.0,0.00,5.53
2,2026-01-05,2588#刺绣收腹提臀裤,1.0,59.90,88.64
3,2026-01-08,2588#刺绣收腹提臀裤,1.0,59.90,12.72
4,2026-01-09,2588#刺绣收腹提臀裤,0.0,0.00,11.62
5,2026-01-10,2588#刺绣收腹提臀裤,1.0,59.90,39.20
6,2026-01-11,2588#刺绣收腹提臀裤,1.0,59.90,19.89
7,2026-01-12,2588#刺绣收腹提臀裤,27.0,1687.20,1031.31
8,2026-01-13,2588#刺绣收腹提臀裤,196.0,11987.00,8043.59
9,2026-01-14,2588#刺绣收腹提臀裤,215.0,12905.50,7561.81


In [119]:
#拼接店铺消耗表

df_consume1 = pd.read_excel("店铺消耗ADQ25号.xlsx")

# 选择需要的列并提取日期
df_consume1 = df_consume1[['日期', '产品名称', '累计消耗金额', '成交订单数','成交金额']]
df_consume1['下单时间'] = df_consume1['日期'].str.extract(r'(\d{4}\.\d{1,2}\.\d{1,2})')[0]
df_consume1['下单时间'] = pd.to_datetime(df_consume1['下单时间'], format='%Y.%m.%d').dt.strftime('%Y-%m-%d')

# 删除原日期列
df_consume1.drop(columns=['日期'], inplace=True)

# 使用pivot_table进行数据透视
df_consume1 = pd.pivot_table(
    df_consume1, 
    index=['下单时间', '产品名称'], 
    values=['累计消耗金额', '成交订单数', '成交金额'], 
    aggfunc=np.sum
).reset_index()

# 将下单时间转换为日期格式
df_consume1["下单时间"] = pd.to_datetime(df_consume1["下单时间"], format='%Y-%m-%d')

# 重命名列
df_consume1.rename(columns={'成交订单数': 'ADQ成交订单数', '累计消耗金额': 'ADQ累计消耗金额', '成交金额': 'ADQ广告gmv'}, inplace=True)


print(df_consume1)

          下单时间          产品名称  ADQ成交订单数   ADQ广告gmv  ADQ累计消耗金额
0   2026-01-01  2588#刺绣收腹提臀裤       0.0       0.00       2.43
1   2026-01-02  2588#刺绣收腹提臀裤       0.0       0.00       1.29
2   2026-01-03  2588#刺绣收腹提臀裤       0.0       0.00       4.48
3   2026-01-04  2588#刺绣收腹提臀裤       0.0       0.00       7.28
4   2026-01-05  2588#刺绣收腹提臀裤       1.0      59.90      29.23
5   2026-01-06  2588#刺绣收腹提臀裤       0.0       0.00       0.09
6   2026-01-08  2588#刺绣收腹提臀裤       0.0       0.00   10562.49
7   2026-01-09  2588#刺绣收腹提臀裤       5.0     299.50    1895.80
8   2026-01-10  2588#刺绣收腹提臀裤     113.0    6778.70    7831.28
9   2026-01-11  2588#刺绣收腹提臀裤     485.0   29185.40   27628.74
10  2026-01-12  2588#刺绣收腹提臀裤     991.0   59697.64   35441.18
11  2026-01-13  2588#刺绣收腹提臀裤     621.0   37309.70   21267.49
12  2026-01-14  2588#刺绣收腹提臀裤     308.0   18486.21   11394.29
13  2026-01-15  2588#刺绣收腹提臀裤     182.0   10906.80    5404.39
14  2026-01-16  2588#刺绣收腹提臀裤     222.0   13377.70    7105.25
15  2026-01-17  2588#刺绣收

In [120]:
final = pd.merge(result,df_consume,on = ['下单时间','产品名称'],how = 'left')


final = final[['下单时间','产品名称','成交订单数','下单单量','支付单量','发货单量','已签收','发货中1','发货中2','售后中','已退款','发货前退款单量',"预计签收单量","累计消耗金额",'运费险预计投保费用','支付金额','成交金额','发货时长（小时）','单件支付单量','双件支付单量','单件签收单量','双件签收单量','单件发货前退款单量','双件发货前退款单量']]
final['发货中'] = final['发货中1'] + final['发货中2']
final = final[['下单时间','产品名称','成交订单数','下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量","累计消耗金额",'运费险预计投保费用','支付金额','成交金额','发货时长（小时）','单件支付单量','双件支付单量','单件签收单量','双件签收单量','单件发货前退款单量','双件发货前退款单量']]
final.rename(columns={"成交订单数":"广告单量"},inplace=True)

final.rename(columns={'累计消耗金额': '微信豆累计消耗金额', '广告单量': '微信豆广告单量','成交金额': '微信豆广告gmv','运费险预计投保费用': '运费险'}, inplace=True)

df_consume1['下单时间'] = df_consume1['下单时间'].astype(str)
final = pd.merge(final, df_consume1, on=['下单时间', '产品名称'], how='left')
final['全店支付gmv'] = final['支付金额']
final['发货时长'] = final['发货时长（小时）'] / final['支付单量']

final = final[['下单时间','运费险','产品名称','ADQ成交订单数','微信豆广告单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量","ADQ累计消耗金额","微信豆累计消耗金额",'全店支付gmv','微信豆广告gmv','ADQ广告gmv','支付金额','发货时长（小时）','发货时长','单件支付单量','双件支付单量','单件签收单量','双件签收单量','单件发货前退款单量','双件发货前退款单量']]

# 消耗为空的赋值0
final.fillna(0, inplace=True)



In [121]:
def concat_unique_strings(x):
    # 步骤1：移除NaN值
    cleaned = x.dropna()
    if cleaned.empty:
        return np.nan
    
    # 步骤2：使用集合去重（不保留顺序）
    # return ','.join(set(cleaned))
    
    # 步骤3：去重且保留首次出现顺序
    seen = set()
    unique_list = [item for item in cleaned if not (item in seen or seen.add(item))]
    return ','.join(unique_list)

# 创建智能聚合字典
agg_dict = {}
for column in final.columns:
    if column == '下单时间':  # 跳过下单时间列
        continue
        
    if pd.api.types.is_numeric_dtype(final[column]):
        agg_dict[column] = 'sum'  # 数值列求和
    else:
        agg_dict[column] = concat_unique_strings  # 字符串列去重拼接


# 执行分组聚合
final_sum = final.groupby('下单时间', as_index=False).agg(agg_dict)


final['平均客单'] = final['支付金额'] / final['支付单量']
final_sum['平均客单'] = final_sum['支付金额'] / final_sum['支付单量']
final = final[['下单时间','运费险','产品名称','ADQ成交订单数','微信豆广告单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量","ADQ累计消耗金额","微信豆累计消耗金额",'全店支付gmv','微信豆广告gmv','ADQ广告gmv','支付金额','平均客单','发货时长','单件支付单量','双件支付单量','单件签收单量','双件签收单量','单件发货前退款单量','双件发货前退款单量']]
final_sum = final_sum[['下单时间','运费险','产品名称','ADQ成交订单数','微信豆广告单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量","ADQ累计消耗金额","微信豆累计消耗金额",'全店支付gmv','微信豆广告gmv','ADQ广告gmv','支付金额','平均客单','发货时长','单件支付单量','双件支付单量','单件签收单量','双件签收单量','单件发货前退款单量','双件发货前退款单量']]


In [122]:
final_sum

,下单时间,运费险,产品名称,ADQ成交订单数,微信豆广告单量,支付单量,发货单量,已签收,发货中,售后中,...,ADQ广告gmv,支付金额,平均客单,发货时长,单件支付单量,双件支付单量,单件签收单量,双件签收单量,单件发货前退款单量,双件发货前退款单量
0,2026-10-28,关闭,2588#刺绣收腹提臀裤,211.0,0.0,233,221,165,2,0,...,12678.9,14016.7,60.157511,0.121195,0,233,0,167,0,12
1,2026-10-29,关闭,2588#刺绣收腹提臀裤,209.0,0.0,221,208,162,0,0,...,12569.1,13297.9,60.171493,0.138142,0,221,0,162,0,13
2,2026-10-30,关闭,2588#刺绣收腹提臀裤,190.0,0.0,210,198,163,1,1,...,11451.0,12649.0,60.233333,0.110540,0,210,0,164,0,12
3,2026-10-31,关闭,2588#刺绣收腹提臀裤,207.0,0.0,233,217,179,3,2,...,12409.3,13986.7,60.028755,0.119388,0,233,0,182,0,16
4,2026-11-01,关闭,2588#刺绣收腹提臀裤,193.0,0.0,209,200,153,2,0,...,11620.7,12579.1,60.187081,0.123853,0,209,0,155,0,9
5,2026-11-02,关闭,2588#刺绣收腹提臀裤,141.0,0.0,159,149,112,0,0,...,8515.8,9554.1,60.088679,0.114720,0,159,0,112,0,10
6,2026-11-03,关闭,2588#刺绣收腹提臀裤,221.0,0.0,244,230,185,1,1,...,13447.6,14949.0,61.266393,0.128603,0,244,0,186,0,14
7,2026-11-04,关闭,2588#刺绣收腹提臀裤,121.0,0.0,148,138,103,0,3,...,7277.9,8895.2,60.102703,0.117502,0,148,0,103,0,10
8,2026-11-05,关闭,2588#刺绣收腹提臀裤,112.0,0.0,138,124,103,0,0,...,6728.8,8326.1,60.334058,0.113502,0,138,0,103,0,14
9,2026-11-06,关闭,2588#刺绣收腹提臀裤,184.0,0.0,207,197,164,1,0,...,11091.6,12479.3,60.286473,0.130824,0,207,0,165,0,10


In [123]:
# writer = pd.ExcelWriter('直播汇总.xlsx')
# final.to_excel(writer, sheet_name='汇总', index=False)
# writer.save()
writer = pd.ExcelWriter('直播汇总.xlsx')
final_sum.to_excel(writer, sheet_name='汇总', index=False)
final.to_excel(writer, sheet_name='产品分类', index=False)
writer.close()  # 改用 close() 方法

In [124]:
df = pd.read_excel('直播汇总.xlsx', sheet_name='汇总')  # 读取 Excel 文件中的汇总表例如，从 CSV 文件加载数据
len(df)

22

In [125]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import column_index_from_string

def write_data_to_range(ws, data, 开始行, 开始列, 结束行, 结束列):
    """
    将数据写入Excel工作表指定范围
    
    参数:
        ws: Excel工作表对象
        data: 二维数据列表
        开始行: 起始行号(整数)
        开始列: 起始列字母(如'A', 'B')
        结束行: 结束行号(整数)
        结束列: 结束列字母(如'C', 'D')
    """
    # 将字母列转换为数字索引
    start_col_num = column_index_from_string(开始列)
    end_col_num = column_index_from_string(结束列)
    
    # 遍历指定范围
    for row_idx in range(开始行, 结束行 + 1):
        for col_idx in range(start_col_num, end_col_num + 1):
            # 计算数据中的位置
            data_row = row_idx - 开始行
            data_col = col_idx - start_col_num
            
            # 检查数据索引是否有效
            if data_row < len(data) and data_col < len(data[data_row]):
                value = data[data_row][data_col]
                ws.cell(row=row_idx, column=col_idx, value=value)

df = pd.read_excel('直播汇总.xlsx', sheet_name='汇总')  # 读取 Excel 文件中的汇总表例如，从 CSV 文件加载数据

product_name = df.loc[0, '产品名称']

# 指定要写入的列和前20行数据
columns_to_write = ['下单时间', '运费险', 'ADQ成交订单数', '微信豆广告单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量","ADQ累计消耗金额","微信豆累计消耗金额",'全店支付gmv','微信豆广告gmv','ADQ广告gmv','支付金额']
if len(df) > 20:
    data_to_write = df[columns_to_write].head(20)
    df_length = 20
else:
    df_length = len(df)
    data_to_write = df[columns_to_write]

# Excel 文件路径
excel_file = './视频号_签收记录表自动化测试.xlsx'

try:
    wb = load_workbook(excel_file)
    
    # 确保至少有一个可见的工作表
    if all(sheet.sheet_state == 'hidden' for sheet in wb.worksheets):
        wb.worksheets[0].sheet_state = 'visible'
    
    for sheet in wb.worksheets:
        if sheet.title != "原始表格":
            del wb[sheet.title]
    
    # 复制或创建输出表格
    sheet_name = '原始表格'
    
    if sheet_name in wb.sheetnames:
        # 获取原始表格
        original_ws = wb[sheet_name]
        
        # 创建输出表格（复制原始表格）
        output_ws = wb.copy_worksheet(original_ws)
        output_ws.title = "输出表格"
        
        # 将输出表格移动到第一个位置
        sheets = wb.worksheets
        output_index = sheets.index(output_ws)
        if output_index > 0:
            sheets.pop(output_index)
            sheets.insert(0, output_ws)
            wb._sheets = sheets
        
        # 现在使用输出表格
        ws = output_ws
    else:
        # 创建新的输出表格
        ws = wb.create_sheet("输出表格")
        
        # 移动到第一个位置
        wb.move_sheet(ws, offset=-len(wb.sheetnames))
    
    # 准备数据（转换为列表格式）
    data = data_to_write.values.tolist()
    print(data)

    # 写入数据到 B4-E23 范围 (4列)
    write_data_to_range(
        ws,
        [row[:4] for row in data],
        开始行=4,
        开始列="B",
        结束行=23,
        结束列="E"
    )
    # 写入数据到 E4-P23 范围 (8列)
    write_data_to_range(
        ws,
        [row[4:12] for row in data],
        开始行=4,
        开始列="I",
        结束行=23,
        结束列="P"
    )
    # 写入数据到 W4-AA23 范围 (2列)
    write_data_to_range(
        ws,
        [row[12:17] for row in data],
        开始行=4,
        开始列="W",
        结束行=23,
        结束列="AA"
    )
    write_data_to_range(
        ws,
        [row[17:18] for row in data],
        开始行=4,
        开始列="AE",
        结束行=23,
        结束列="AF"
    )
    # 保存工作簿
    wb.save(excel_file)
    print("数据写入成功！")
    
except PermissionError:
    print("错误：文件被其他程序锁定，请关闭后重试")
except FileNotFoundError:
    print(f"错误：文件 {excel_file} 不存在")
except Exception as e:
    print(f"处理文件时出错: {e}")


[['2026-10-28', '关闭', 211, 0, 233, 221, 165, 2, 0, 54, 12, 167, 7193.69, 0, 14016.69999999994, 0, 12678.9, 14016.69999999994], ['2026-10-29', '关闭', 209, 0, 221, 208, 162, 0, 0, 46, 13, 162, 6587.660000000001, 0, 13297.89999999995, 0, 12569.1, 13297.89999999995], ['2026-10-30', '关闭', 190, 0, 210, 198, 163, 1, 1, 33, 12, 164, 5895.55, 0, 12648.99999999995, 0, 11451.0, 12648.99999999995], ['2026-10-31', '关闭', 207, 0, 233, 217, 179, 3, 2, 33, 16, 182, 5735.46, 0, 13986.69999999994, 0, 12409.3, 13986.69999999994], ['2026-11-01', '关闭', 193, 0, 209, 200, 153, 2, 0, 45, 9, 155, 6035.95, 0, 12579.09999999995, 0, 11620.7, 12579.09999999995], ['2026-11-02', '关闭', 141, 0, 159, 149, 112, 0, 0, 37, 10, 112, 4117.829999999999, 0, 9554.099999999971, 0, 8515.8, 9554.099999999971], ['2026-11-03', '关闭', 221, 0, 244, 230, 185, 1, 1, 43, 14, 186, 6864.66, 0, 14948.99999999994, 0, 13447.6, 14948.99999999994], ['2026-11-04', '关闭', 121, 0, 148, 138, 103, 0, 3, 32, 10, 103, 3635.21, 0, 8895.199999999977, 0, 72

In [126]:
data

[['2026-10-28',
  '关闭',
  211,
  0,
  233,
  221,
  165,
  2,
  0,
  54,
  12,
  167,
  7193.69,
  0,
  14016.69999999994,
  0,
  12678.9,
  14016.69999999994],
 ['2026-10-29',
  '关闭',
  209,
  0,
  221,
  208,
  162,
  0,
  0,
  46,
  13,
  162,
  6587.660000000001,
  0,
  13297.89999999995,
  0,
  12569.1,
  13297.89999999995],
 ['2026-10-30',
  '关闭',
  190,
  0,
  210,
  198,
  163,
  1,
  1,
  33,
  12,
  164,
  5895.55,
  0,
  12648.99999999995,
  0,
  11451.0,
  12648.99999999995],
 ['2026-10-31',
  '关闭',
  207,
  0,
  233,
  217,
  179,
  3,
  2,
  33,
  16,
  182,
  5735.46,
  0,
  13986.69999999994,
  0,
  12409.3,
  13986.69999999994],
 ['2026-11-01',
  '关闭',
  193,
  0,
  209,
  200,
  153,
  2,
  0,
  45,
  9,
  155,
  6035.95,
  0,
  12579.09999999995,
  0,
  11620.7,
  12579.09999999995],
 ['2026-11-02',
  '关闭',
  141,
  0,
  159,
  149,
  112,
  0,
  0,
  37,
  10,
  112,
  4117.829999999999,
  0,
  9554.099999999971,
  0,
  8515.8,
  9554.099999999971],
 ['2026-11-03',
